# Phishing Email Detection — Exploration

Quick EDA: class balance, text lengths, URL signals, top TF-IDF terms.
Run from project root. Uses only static analysis (never opens URLs).

In [ ]:
import pandas as pd
from pathlib import Path
from src.preprocessing import clean_dataframe
from src.url_features import url_features_to_matrix, FEATURE_NAMES

df = clean_dataframe(pd.read_csv('data/raw/emails.csv'))
print(df['label'].value_counts())
print(df.head(3).to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Class distribution
df['label'].value_counts().plot(kind='bar')
plt.xticks([0, 1], ['Safe', 'Phishing'])
plt.title('Class distribution')
plt.show()

# Text length by class
df['length'] = df['combined_text'].str.len()
sns.histplot(data=df, x='length', hue='label', bins=30)
plt.title('Email length by class')
plt.show()

In [ ]:
# URL feature summary by class
import numpy as np
mat = url_features_to_matrix(df['combined_text'])
url_df = pd.DataFrame(mat, columns=FEATURE_NAMES)
url_df['label'] = df['label'].values
print(url_df.groupby('label').mean().T.to_string())

In [ ]:
# Top TF-IDF terms per class (fit on full data for EXPLORATION only;
# training script fits vectorizer on train split to avoid leakage)
from sklearn.feature_extraction.text import TfidfVectorizer
from src.preprocessing import clean_text
import numpy as np

vec = TfidfVectorizer(preprocessor=clean_text, stop_words='english', max_features=2000)
X = vec.fit_transform(df['combined_text'])
terms = np.array(vec.get_feature_names_out())
for label, name in [(1, 'Phishing'), (0, 'Safe')]:
    idx = df['label'].values == label
    scores = X[idx].mean(axis=0).A1
    top = terms[scores.argsort()[-15:][::-1]]
    print(name, list(top))